In [ ]:
# Setup imports & load dataset
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
import os
df = pd.read_csv('data/raw_transactions.csv', na_values=['Nan', ''])


## Section 12: Advanced concepts (5+ Years Experience)

### Concept 1: Deep Memory usage
**Explanation**: Measures actual RAM usage. Setting `deep=True` inspects dynamically allocated object structures (like strings) rather than just the pointer sizes.

**Syntax**: `df.memory_usage(deep=True)`

In [34]:
print(df.memory_usage(deep=True))

Index                    132
transaction_id        570000
customer_id           550000
transaction_amount     80000
account_age_months     80000
transaction_date      610355
region                535443
dtype: int64


### Concept 2: Numeric Downcasting
**Explanation**: Saves memory by shrinking numerical precision (e.g. float64 to float32 or int64 to uint8) based on minimum/maximum value limits in the column.

**Syntax**: `pd.to_numeric(df['col'], downcast='unsigned|float')`

In [35]:
print(pd.to_numeric(df['account_age_months'], downcast='unsigned').head(5))

0    103
1     62
2     71
3      0
4     10
Name: account_age_months, dtype: uint8


### Concept 3: Categorical Conversion
**Explanation**: Optimizes redundant string storage by mapping string categories to small integer codes, drastically reducing memory usage.

**Syntax**: `df['col'].astype('category')`

In [36]:
print(df['region'].astype('category').head(5))

0    North
1    North
2    South
3     West
4    North
Name: region, dtype: category
Categories (12, object): [' East ', ' North ', ' South ', ' West ', ..., 'east', 'north', 'south', 'west']


### Concept 4: Chunksize Reading
**Explanation**: Ingests files incrementally by returning a generator (TextFileReader). This prevents Out-Of-Memory errors when loading files larger than system RAM.

**Syntax**: `pd.read_csv('file.csv', chunksize=size)`

In [37]:
for chunk in pd.read_csv('data/raw_transactions.csv', chunksize=1000):
    print(f"Chunk columns: {chunk.columns.tolist()}, rows: {len(chunk)}")
    break

Chunk columns: ['transaction_id', 'customer_id', 'transaction_amount', 'account_age_months', 'transaction_date', 'region'], rows: 1000


### Concept 5: Groupby Transform
**Explanation**: Calculates group-level metrics but broadcasts them back to match the original index shape. Useful for group scaling, normalization, or filling group NaNs.

**Syntax**: `df.groupby('grp')['val'].transform('func')`

In [38]:
df_temp = df.copy()
df_temp['region_avg'] = df_temp.groupby('region')['transaction_amount'].transform('mean')
print(df_temp[['region', 'transaction_amount', 'region_avg']].head(5))

  region  transaction_amount  region_avg
0  North                 NaN  777.025695
1  North              931.17  777.025695
2  South              932.88  750.832493
3   West              339.48  742.332103
4  North              100.39  777.025695


### Concept 6: Groupby Filter
**Explanation**: Filters out entire group partitions that do not satisfy an aggregate boolean criteria (e.g. dropping low-volume categories).

**Syntax**: `df.groupby('grp').filter(lambda x: condition)`

In [39]:
filtered = df.groupby('region').filter(lambda x: len(x) > 1000)
print("Regions with >1000 txs:", filtered['region'].unique())

Regions with >1000 txs: ['North' 'South' 'West' 'East']


### Concept 7: Groupby Custom Apply
**Explanation**: Executes a user-defined function independently on each split group DataFrame, and combines the results back into a single structure.

**Syntax**: `df.groupby('grp').apply(lambda g: func(g))`

In [40]:
print(df.groupby('region').apply(lambda g: g.nlargest(1, 'transaction_amount')))

             transaction_id customer_id  transaction_amount  \
region                                                        
 East   9384       TX108076      C93227             1487.22   
 North  3153       TX105486      C54473             1435.45   
 South  6784       TX109389      C25014             1489.00   
 West   1740       TX105630      C17100             1446.51   
East    1685       TX104220      C32241             1499.74   
North   4996       TX103565      C27146             1499.19   
South   4457       TX107105      C79514             1499.83   
West    5438       TX105317      C45992             1499.73   
east    4254       TX102864      C80010             1498.41   
north   6209       TX101576      C82692             1296.83   
south   4142       TX103729      C97062             1496.91   
west    2704       TX100292      C31174             1444.80   

              account_age_months transaction_date   region  
region                                                  

C:\Users\DELL\AppData\Local\Temp\ipykernel_9452\2310601459.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  print(df.groupby('region').apply(lambda g: g.nlargest(1, 'transaction_amount')))


### Concept 8: Rolling windows
**Explanation**: Computes aggregations over a sliding window index range. Standard for calculating moving averages, smoothing signals, or tracking financial trends.

**Syntax**: `df['col'].rolling(window=size).mean()`

In [41]:
df_dates = df.copy()
df_dates['transaction_date'] = pd.to_datetime(df_dates['transaction_date'], format='mixed')
df_dates = df_dates.set_index('transaction_date').sort_index()
print(df_dates['transaction_amount'].rolling(window=10).mean().head(15))

transaction_date
2025-01-01 00:00:00         NaN
2025-01-01 00:00:00         NaN
2025-01-01 00:00:00         NaN
2025-01-01 00:00:00         NaN
2025-01-01 00:00:00         NaN
2025-01-01 00:00:00         NaN
2025-01-01 00:00:00         NaN
2025-01-01 00:00:00         NaN
2025-01-01 03:31:09         NaN
2025-01-01 03:43:42    1033.165
2025-01-02 00:00:00     963.215
2025-01-02 00:00:00     955.245
2025-01-02 00:00:00     966.548
2025-01-02 00:00:00     833.701
2025-01-02 00:00:00     932.605
Name: transaction_amount, dtype: float64


### Concept 9: Expanding windows
**Explanation**: Computes aggregate values starting from the index inception, expanding the window size by one at each step. Perfect for running cumulative statistics.

**Syntax**: `df['col'].expanding().sum()`

In [42]:
print(df_dates['transaction_amount'].expanding().sum().head(10))

transaction_date
2025-01-01 00:00:00     1133.71
2025-01-01 00:00:00     1986.09
2025-01-01 00:00:00     2637.13
2025-01-01 00:00:00     4124.00
2025-01-01 00:00:00     4480.24
2025-01-01 00:00:00     5796.65
2025-01-01 00:00:00     7249.45
2025-01-01 00:00:00     8465.47
2025-01-01 03:31:09     9596.68
2025-01-01 03:43:42    10331.65
Name: transaction_amount, dtype: float64


### Concept 10: EWM
**Explanation**: Computes Exponentially Weighted metrics, assigning exponentially higher weights to recent observations based on decay parameters.

**Syntax**: `df['col'].ewm(alpha=alpha).mean()`

In [43]:
print(df_dates['transaction_amount'].ewm(alpha=0.1).mean().head(10))

transaction_date
2025-01-01 00:00:00    1133.710000
2025-01-01 00:00:00     985.641579
2025-01-01 00:00:00     862.172362
2025-01-01 00:00:00    1043.823318
2025-01-01 00:00:00     875.919404
2025-01-01 00:00:00     969.929037
2025-01-01 00:00:00    1062.485697
2025-01-01 00:00:00    1089.443636
2025-01-01 03:31:09    1096.261749
2025-01-01 03:43:42    1040.791184
Name: transaction_amount, dtype: float64


### Concept 11: Lag and Shift
**Explanation**: Shifts indices by positive (lag) or negative (lead) offsets. Crucial for autoregressive feature design or step-by-step delta calculations.

**Syntax**: `df['col'].shift(periods)` / `df['col'].pct_change()`

In [44]:
df_lag = df[['transaction_id', 'transaction_amount']].copy()
df_lag['prev_amount'] = df_lag['transaction_amount'].shift(1)
print(df_lag.head(5))

  transaction_id  transaction_amount  prev_amount
0       TX104873                 NaN          NaN
1       TX107538              931.17          NaN
2       TX101583              932.88       931.17
3       TX101864              339.48       932.88
4       TX104799              100.39       339.48


### Concept 12: MultiIndex Slicing
**Explanation**: Slices multiple dimensions of indexed indices cleanly using `pd.IndexSlice` to target specific combinations of levels.

**Syntax**: `df.loc[pd.IndexSlice[level1_val, level2_val], :]`

In [45]:
df_mi = df.set_index(['region', 'customer_id']).sort_index()
print(df_mi.loc[pd.IndexSlice['North', :], :].head(3))

                   transaction_id  transaction_amount  account_age_months  \
region customer_id                                                          
North  C10053            TX102381              348.73                  40   
       C10053            TX105576             1332.54                  26   
       C10074            TX109222              972.66                  89   

                       transaction_date  
region customer_id                       
North  C10053                11/02/2026  
       C10053       2025-04-22 19:41:22  
       C10074                01-18-2026  


### Concept 13: Cross-sections
**Explanation**: Retrieves a specific level slice from MultiIndexed DataFrames, providing a cleaner notation than `IndexSlice`.

**Syntax**: `df.xs(key, level)`

In [46]:
print(df_mi.xs(key='North', level='region').head(3))

            transaction_id  transaction_amount  account_age_months  \
customer_id                                                          
C10053            TX102381              348.73                  40   
C10053            TX105576             1332.54                  26   
C10074            TX109222              972.66                  89   

                transaction_date  
customer_id                       
C10053                11/02/2026  
C10053       2025-04-22 19:41:22  
C10074                01-18-2026  


### Concept 14: Stack and Unstack
**Explanation**: Stacking pivots column labels into index rows, and unstacking pivots index levels into column header layers.

**Syntax**: `df.unstack(level)` / `df.stack()`

In [47]:
df_un = df.pivot_table(index='region', columns='account_age_months', values='transaction_amount', aggfunc='mean')
print("Stacked Result:\n", df_un.stack().head(10))

Stacked Result:
 region  account_age_months
East    5                      981.945
        7                     1214.980
        10                    1319.620
        11                     855.180
        15                     664.470
        17                     213.220
        18                     440.170
        19                     779.480
        20                     806.770
        21                    1144.720
dtype: float64


### Concept 15: Non-Exact Merging
**Explanation**: Merges datasets on matching keys that do not align exactly (usually time). Finds the nearest match backwards or forwards in time.

**Syntax**: `pd.merge_asof(df_left, df_right, on='key', direction='backward|forward')`

In [48]:
df_t_sorted = df_dates.reset_index()[['transaction_date', 'transaction_amount']].sort_values('transaction_date')
df_rates = pd.DataFrame({
    'rate_date': pd.to_datetime(['2025-01-01', '2025-06-01', '2026-01-01']),
    'exchange_rate': [1.0, 1.05, 1.10]
}).sort_values('rate_date')

merged_rates = pd.merge_asof(
    df_t_sorted,
    df_rates,
    left_on='transaction_date',
    right_on='rate_date',
    direction='backward'
)
print(merged_rates.head(5))

  transaction_date  transaction_amount  rate_date  exchange_rate
0       2025-01-01             1133.71 2025-01-01            1.0
1       2025-01-01              852.38 2025-01-01            1.0
2       2025-01-01              651.04 2025-01-01            1.0
3       2025-01-01             1486.87 2025-01-01            1.0
4       2025-01-01              356.24 2025-01-01            1.0


### Concept 16: Explode
**Explanation**: Expands list-like values in a single cell into individual rows, duplicating index values for all other columns.

**Syntax**: `df.explode('list_col')`

In [49]:
df_exp = pd.DataFrame({'customer_id': ['C12'], 'regions': [['North', 'East']]})
print(df_exp.explode('regions'))

  customer_id regions
0         C12   North
0         C12    East


### Concept 17: Regex Capture str.extract
**Explanation**: Extracts regular expression capture groups directly into individual columns, facilitating structured text parsing.

**Syntax**: `df['col'].str.extract(r'regex_pattern')`

In [50]:
print(df['customer_id'].str.extract(r'C(\d+)').head(5))

       0
0  54313
1  40161
2  91183
3  36071
4  72296
